# Jane Street 模型训练 - 神经网络(MLP)

## 为什么选择神经网络？

### 神经网络的优势

**1. 非线性建模能力强**
- 可以学习复杂的非线性关系
- 不需要手动进行特征变换
- 能够自动发现特征之间的交互作用

**2. 端到端学习**
- 从原始特征直接学习到目标
- 不需要复杂的特征工程
- 可以整合多种类型的特征

**3. 持续学习**
- 可以在新数据上继续训练
- 适应数据分布的变化
- 支持在线学习

### 神经网络 vs GBDT

| 特性 | 神经网络 | GBDT (LightGBM/XGBoost) |
|------|----------|---------------------------|
| 数据类型 | 任意结构 | 表格数据 |
| 特征工程 | 需要较少 | 需要较多 |
| 训练时间 | 较长 | 较短 |
| 调参难度 | 较高 | 中等 |
| 解释性 | 较低 | 较高 |
| 非线性能力 | 很强 | 强 |
| 过拟合风险 | 较高 | 中等 |

**何时使用神经网络：**
- GBDT已经达到瓶颈
- 特征之间存在复杂的非线性关系
- 有足够的数据和计算资源
- 追求极致的性能

### MLP架构说明

**MLP (多层感知机)** 是最基础的神经网络架构：
- 由多个全连接层组成
- 每层后接激活函数
- 使用批归一化和Dropout防止过拟合
- 适合处理表格数据

## 1. 环境设置

In [ ]:
import numpy as np
import pandas as pd
import polars as pl
from pathlib import Path
import warnings
import gc
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

# 设置路径
# 使用相对路径
DATA_PATH = Path('../Dataset')
PROCESSED_PATH = Path('./processed_data')
MODEL_OUTPUT_PATH = Path('./models')

# 创建模型输出目录
MODEL_OUTPUT_PATH.mkdir(exist_ok=True, parents=True)

# 设置随机种子
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# 检查设备
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用设备: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU名称: {torch.cuda.get_device_name(0)}")
    print(f"GPU内存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## 2. 配置参数

In [ ]:
class CONFIG:
    # 数据路径
    TRAIN_FILE = PROCESSED_PATH / 'training.parquet'
    VALID_FILE = PROCESSED_PATH / 'validation.parquet'
    
    # 特征列
    FEATURE_COLS = [f"feature_{i:02d}" for i in range(79)] + [f"responder_{i}_lag_1" for i in range(9)]
    
    # 目标列
    TARGET_COL = "responder_6"
    
    # 权重列
    WEIGHT_COL = "weight"
    
    # 训练参数
    SEED = 42
    N_FOLDS = 5
    BATCH_SIZE = 8192
    MAX_EPOCHS = 100
    PATIENCE = 10
    
    # 网络架构
    HIDDEN_DIMS = [512, 256, 128]  # 隐藏层维度
    DROPOUT = [0.2, 0.2, 0.1]  # 每层的Dropout率
    USE_BATCH_NORM = True  # 是否使用批归一化
    
    # 优化器参数
    LEARNING_RATE = 1e-3
    WEIGHT_DECAY = 1e-4
    
    # 学习率调度
    LR_SCHEDULER_PATIENCE = 5
    LR_SCHEDULER_FACTOR = 0.5
    
print("=== 训练配置 ===")
print(f"特征数量: {len(CONFIG.FEATURE_COLS)}")
print(f"目标列: {CONFIG.TARGET_COL}")
print(f"网络架构: {len(CONFIG.FEATURE_COLS)} -> {' -> '.join(map(str, CONFIG.HIDDEN_DIMS))} -> 1")
print(f"批大小: {CONFIG.BATCH_SIZE}")
print(f"最大轮数: {CONFIG.MAX_EPOCHS}")

## 3. 自定义数据集和数据加载器

In [ ]:
class JaneStreetDataset(Dataset):
    """
    Jane Street数据集类
    
    使用延迟加载策略，在__getitem__时才转换为tensor
    """
    
    def __init__(self, df, feature_cols, target_col, weight_col):
        self.df = df.reset_index(drop=True)
        self.feature_cols = feature_cols
        self.target_col = target_col
        self.weight_col = weight_col
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        # 获取特征
        features = self.df.loc[idx, self.feature_cols].values.astype(np.float32)
        
        # 获取目标
        target = self.df.loc[idx, self.target_col].astype(np.float32)
        
        # 获取权重
        weight = self.df.loc[idx, self.weight_col].astype(np.float32)
        
        return {
            'features': torch.from_numpy(features),
            'target': torch.tensor(target, dtype=torch.float32),
            'weight': torch.tensor(weight, dtype=torch.float32)
        }

def create_data_loaders(train_df, valid_df, batch_size):
    """
    创建数据加载器
    """
    train_dataset = JaneStreetDataset(
        train_df, CONFIG.FEATURE_COLS, CONFIG.TARGET_COL, CONFIG.WEIGHT_COL
    )
    valid_dataset = JaneStreetDataset(
        valid_df, CONFIG.FEATURE_COLS, CONFIG.TARGET_COL, CONFIG.WEIGHT_COL
    )
    
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,  # Mac上可能需要设为0
        pin_memory=False
    )
    
    valid_loader = DataLoader(
        valid_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=False
    )
    
    return train_loader, valid_loader

print("数据加载器类已定义")

## 4. 神经网络模型定义

In [ ]:
class MLPModel(nn.Module):
    """
    多层感知机模型
    
    架构:
    - 输入层
    - 多个隐藏层（带批归一化、激活函数、Dropout）
    - 输出层
    """
    
    def __init__(self, input_dim, hidden_dims, dropout, use_batch_norm=True):
        super(MLPModel, self).__init__()
        
        self.layers = nn.ModuleList()
        self.batch_norms = nn.ModuleList() if use_batch_norm else None
        self.dropouts = nn.ModuleList()
        
        # 构建网络
        dims = [input_dim] + hidden_dims
        
        for i in range(len(dims) - 1):
            # 线性层
            self.layers.append(nn.Linear(dims[i], dims[i + 1]))
            
            # 批归一化
            if use_batch_norm:
                self.batch_norms.append(nn.BatchNorm1d(dims[i + 1]))
            
            # Dropout
            if i < len(dropout):
                self.dropouts.append(nn.Dropout(dropout[i]))
            else:
                self.dropouts.append(nn.Dropout(0))
        
        # 输出层
        self.output_layer = nn.Linear(hidden_dims[-1], 1)
        
        # 激活函数
        self.activation = nn.SiLU()  # Swish激活函数
        
        # 输出激活（限制范围）
        self.output_activation = nn.Tanh()
    
    def forward(self, x):
        # 前向传播
        for i, (layer, dropout) in enumerate(zip(self.layers, self.dropouts)):
            x = layer(x)
            
            # 批归一化
            if self.batch_norms is not None and i < len(self.batch_norms):
                x = self.batch_norms[i](x)
            
            # 激活函数（除最后一层）
            if i < len(self.layers) - 1:
                x = self.activation(x)
            
            # Dropout
            x = dropout(x)
        
        # 输出层
        x = self.output_layer(x)
        x = self.output_activation(x)
        
        # 乘以5以扩展输出范围
        return x * 5

# 测试模型
input_dim = len(CONFIG.FEATURE_COLS)
model = MLPModel(
    input_dim=input_dim,
    hidden_dims=CONFIG.HIDDEN_DIMS,
    dropout=CONFIG.DROPOUT,
    use_batch_norm=CONFIG.USE_BATCH_NORM
).to(DEVICE)

# 计算参数数量
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n模型架构:")
print(model)
print(f"\n总参数数量: {total_params:,}")
print(f"可训练参数数量: {trainable_params:,}")

# 测试前向传播
dummy_input = torch.randn(10, input_dim).to(DEVICE)
dummy_output = model(dummy_input)
print(f"\n测试输入形状: {dummy_input.shape}")
print(f"测试输出形状: {dummy_output.shape}")

## 5. 评估指标和损失函数

In [ ]:
def weighted_r2_score(y_true, y_pred, sample_weight):
    """
    计算加权R²得分
    """
    numerator = np.average((y_true - y_pred) ** 2, weights=sample_weight)
    denominator = np.average(y_true ** 2, weights=sample_weight) + 1e-38
    r2 = 1 - numerator / denominator
    return r2

def weighted_mse_loss(y_pred, y_true, weight):
    """
    加权MSE损失函数
    """
    loss = F.mse_loss(y_pred, y_true, reduction='none')
    weighted_loss = loss * weight
    return weighted_loss.mean()

# 测试损失函数
y_true = torch.randn(100, 1)
y_pred = torch.randn(100, 1)
weight = torch.rand(100, 1)

loss = weighted_mse_loss(y_pred, y_true, weight)
print(f"测试损失: {loss.item():.6f}")
print("\n损失函数已定义")

## 6. 训练和验证函数

In [ ]:
def train_epoch(model, dataloader, optimizer, device):
    """
    训练一个epoch
    """
    model.train()
    total_loss = 0
    all_preds = []
    all_targets = []
    all_weights = []
    
    for batch in dataloader:
        features = batch['features'].to(device)
        targets = batch['target'].to(device).unsqueeze(1)
        weights = batch['weight'].to(device).unsqueeze(1)
        
        optimizer.zero_grad()
        
        # 前向传播
        predictions = model(features)
        
        # 计算损失
        loss = weighted_mse_loss(predictions, targets, weights)
        
        # 反向传播
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        # 收集预测结果
        all_preds.append(predictions.detach().cpu().numpy())
        all_targets.append(targets.cpu().numpy())
        all_weights.append(weights.cpu().numpy())
    
    # 计算R²
    all_preds = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)
    all_weights = np.concatenate(all_weights)
    
    r2 = weighted_r2_score(all_targets, all_preds, all_weights)
    avg_loss = total_loss / len(dataloader)
    
    return avg_loss, r2

def validate(model, dataloader, device):
    """
    验证模型
    """
    model.eval()
    total_loss = 0
    all_preds = []
    all_targets = []
    all_weights = []
    
    with torch.no_grad():
        for batch in dataloader:
            features = batch['features'].to(device)
            targets = batch['target'].to(device).unsqueeze(1)
            weights = batch['weight'].to(device).unsqueeze(1)
            
            # 前向传播
            predictions = model(features)
            
            # 计算损失
            loss = weighted_mse_loss(predictions, targets, weights)
            
            total_loss += loss.item()
            
            # 收集预测结果
            all_preds.append(predictions.cpu().numpy())
            all_targets.append(targets.cpu().numpy())
            all_weights.append(weights.cpu().numpy())
    
    # 计算R²
    all_preds = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)
    all_weights = np.concatenate(all_weights)
    
    r2 = weighted_r2_score(all_targets, all_preds, all_weights)
    avg_loss = total_loss / len(dataloader)
    
    return avg_loss, r2

print("训练和验证函数已定义")

## 7. 数据加载

In [ ]:
def load_data():
    """
    加载训练和验证数据
    """
    
    if not CONFIG.TRAIN_FILE.exists():
        print("警告: 处理后的训练数据不存在!")
        return None, None
    
    required_cols = CONFIG.FEATURE_COLS + [CONFIG.TARGET_COL, CONFIG.WEIGHT_COL, 'date_id']
    
    print("加载训练数据...")
    train_pl = pl.read_parquet(CONFIG.TRAIN_FILE)
    available_cols = [col for col in required_cols if col in train_pl.columns]
    train_pl = train_pl.select(available_cols)
    train_df = train_pl.to_pandas()
    
    # 处理缺失值
    for col in CONFIG.FEATURE_COLS:
        if col in train_df.columns:
            train_df[col] = train_df[col].fillna(0)
    
    print(f"训练数据形状: {train_df.shape}")
    print(f"日期范围: {train_df['date_id'].min()} 到 {train_df['date_id'].max()}")
    
    if CONFIG.VALID_FILE.exists():
        print("加载验证数据...")
        valid_pl = pl.read_parquet(CONFIG.VALID_FILE)
        available_cols = [col for col in required_cols if col in valid_pl.columns]
        valid_pl = valid_pl.select(available_cols)
        valid_df = valid_pl.to_pandas()
        
        for col in CONFIG.FEATURE_COLS:
            if col in valid_df.columns:
                valid_df[col] = valid_df[col].fillna(0)
        
        print(f"验证数据形状: {valid_df.shape}")
        print(f"日期范围: {valid_df['date_id'].min()} 到 {valid_df['date_id'].max()}")
    else:
        print("验证数据不存在，将从训练数据中分割")
        valid_df = None
    
    return train_df, valid_df

train_df, valid_df = load_data()

## 8. 训练循环

In [ ]:
def train_model(train_df, valid_df=None, fold=0):
    """
    训练神经网络模型
    """
    
    # 分割数据
    if valid_df is None:
        # 从训练数据中分割验证集
        unique_dates = sorted(train_df['date_id'].unique())
        valid_dates = unique_dates[-100:]  # 最后100天作为验证集
        train_dates = [d for d in unique_dates if d not in valid_dates]
        
        train_fold = train_df[train_df['date_id'].isin(train_dates)]
        valid_fold = train_df[train_df['date_id'].isin(valid_dates)]
    else:
        train_fold = train_df
        valid_fold = valid_df
    
    print(f"\n训练集大小: {len(train_fold)}")
    print(f"验证集大小: {len(valid_fold)}")
    
    # 创建数据加载器
    train_loader, valid_loader = create_data_loaders(
        train_fold, valid_fold, CONFIG.BATCH_SIZE
    )
    
    # 创建模型
    input_dim = len(CONFIG.FEATURE_COLS)
    model = MLPModel(
        input_dim=input_dim,
        hidden_dims=CONFIG.HIDDEN_DIMS,
        dropout=CONFIG.DROPOUT,
        use_batch_norm=CONFIG.USE_BATCH_NORM
    ).to(DEVICE)
    
    # 优化器
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=CONFIG.LEARNING_RATE,
        weight_decay=CONFIG.WEIGHT_DECAY
    )
    
    # 学习率调度器
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=CONFIG.LR_SCHEDULER_FACTOR,
        patience=CONFIG.LR_SCHEDULER_PATIENCE,
        verbose=True
    )
    
    # 早停
    best_valid_loss = float('inf')
    best_valid_r2 = -float('inf')
    patience_counter = 0
    
    # 训练历史
    history = {
        'train_loss': [],
        'train_r2': [],
        'valid_loss': [],
        'valid_r2': []
    }
    
    print(f"\n开始训练 折 {fold}...")
    
    for epoch in range(CONFIG.MAX_EPOCHS):
        # 训练
        train_loss, train_r2 = train_epoch(model, train_loader, optimizer, DEVICE)
        
        # 验证
        valid_loss, valid_r2 = validate(model, valid_loader, DEVICE)
        
        # 更新学习率
        scheduler.step(valid_loss)
        
        # 记录历史
        history['train_loss'].append(train_loss)
        history['train_r2'].append(train_r2)
        history['valid_loss'].append(valid_loss)
        history['valid_r2'].append(valid_r2)
        
        # 打印进度
        print(f"Epoch {epoch + 1}/{CONFIG.MAX_EPOCHS} - "
              f"Train Loss: {train_loss:.6f}, Train R²: {train_r2:.6f}, "
              f"Valid Loss: {valid_loss:.6f}, Valid R²: {valid_r2:.6f}")
        
        # 早停检查
        if valid_r2 > best_valid_r2:
            best_valid_r2 = valid_r2
            best_valid_loss = valid_loss
            patience_counter = 0
            
            # 保存最佳模型
            torch.save(model.state_dict(), MODEL_OUTPUT_PATH / f'nn_model_fold{fold}.pt')
        else:
            patience_counter += 1
            if patience_counter >= CONFIG.PATIENCE:
                print(f"\n早停触发于 epoch {epoch + 1}")
                break
    
    # 加载最佳模型
    model.load_state_dict(torch.load(MODEL_OUTPUT_PATH / f'nn_model_fold{fold}.pt'))
    
    print(f"\n训练完成!")
    print(f"最佳验证集 R²: {best_valid_r2:.6f}")
    
    return model, history, best_valid_r2

# 开始训练
if train_df is not None:
    model, history, best_r2 = train_model(train_df, valid_df, fold=0)

## 9. 可视化训练历史

In [ ]:
def plot_training_history(history):
    """
    绘制训练历史
    """
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # 损失曲线
    axes[0].plot(history['train_loss'], label='Train Loss')
    axes[0].plot(history['valid_loss'], label='Valid Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Training and Validation Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # R²曲线
    axes[1].plot(history['train_r2'], label='Train R²')
    axes[1].plot(history['valid_r2'], label='Valid R²')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('R² Score')
    axes[1].set_title('Training and Validation R²')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

if 'history' in locals():
    plot_training_history(history)

## 10. 交叉验证训练（可选）

In [ ]:
def train_with_cv(train_df):
    """
    使用交叉验证训练模型
    """
    
    models = []
    cv_scores = []
    
    unique_dates = sorted(train_df['date_id'].unique())
    
    for fold in range(CONFIG.N_FOLDS):
        print(f"\n{'='*20} 折 {fold + 1}/{CONFIG.N_FOLDS} {'='*20}")
        
        # 分割日期
        valid_dates = unique_dates[fold::CONFIG.N_FOLDS]
        train_dates = [d for d in unique_dates if d not in valid_dates]
        
        # 分割数据
        train_fold = train_df[train_df['date_id'].isin(train_dates)]
        valid_fold = train_df[train_df['date_id'].isin(valid_dates)]
        
        # 训练模型
        model, history, best_r2 = train_model(train_fold, valid_fold, fold)
        
        models.append(model)
        cv_scores.append(best_r2)
        
        # 清理内存
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    
    print("\n" + "="*50)
    print("交叉验证训练完成!")
    print(f"平均验证集 R²: {np.mean(cv_scores):.6f} ± {np.std(cv_scores):.6f}")
    print(f"各折得分: {cv_scores}")
    print("="*50)
    
    return models, cv_scores

# 取消注释以运行交叉验证
# if train_df is not None:
#     models, cv_scores = train_with_cv(train_df)

## 11. 模型评估

In [ ]:
def evaluate_model(model, df, batch_size=8192):
    """
    评估模型
    """
    model.eval()
    
    dataset = JaneStreetDataset(
        df, CONFIG.FEATURE_COLS, CONFIG.TARGET_COL, CONFIG.WEIGHT_COL
    )
    
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0
    )
    
    all_preds = []
    all_targets = []
    all_weights = []
    
    with torch.no_grad():
        for batch in dataloader:
            features = batch['features'].to(DEVICE)
            targets = batch['target'].numpy()
            weights = batch['weight'].numpy()
            
            predictions = model(features).cpu().numpy()
            
            all_preds.append(predictions)
            all_targets.append(targets)
            all_weights.append(weights)
    
    all_preds = np.concatenate(all_preds).ravel()
    all_targets = np.concatenate(all_targets)
    all_weights = np.concatenate(all_weights)
    
    r2 = weighted_r2_score(all_targets, all_preds, all_weights)
    
    return r2, all_preds, all_targets

if 'model' in locals() and train_df is not None:
    # 在训练集上评估
    train_r2, train_preds, train_targets = evaluate_model(model, train_df)
    print(f"训练集 R²: {train_r2:.6f}")
    
    # 在验证集上评估
    if valid_df is not None:
        valid_r2, valid_preds, valid_targets = evaluate_model(model, valid_df)
        print(f"验证集 R²: {valid_r2:.6f}")

## 12. 保存模型

In [ ]:
def save_model_and_info(model, history, best_r2, fold=0):
    """
    保存模型和训练信息
    """
    import json
    
    # 保存模型
    model_path = MODEL_OUTPUT_PATH / f'nn_model_fold{fold}.pt'
    torch.save(model.state_dict(), model_path)
    print(f"模型已保存到: {model_path}")
    
    # 保存训练信息
    training_info = {
        'model_type': 'NeuralNetwork',
        'architecture': {
            'input_dim': len(CONFIG.FEATURE_COLS),
            'hidden_dims': CONFIG.HIDDEN_DIMS,
            'dropout': CONFIG.DROPOUT,
            'use_batch_norm': CONFIG.USE_BATCH_NORM,
        },
        'training_params': {
            'batch_size': CONFIG.BATCH_SIZE,
            'learning_rate': CONFIG.LEARNING_RATE,
            'weight_decay': CONFIG.WEIGHT_DECAY,
            'max_epochs': CONFIG.MAX_EPOCHS,
        },
        'best_r2': float(best_r2),
        'history': {k: [float(v) for v in vals] for k, vals in history.items()}
    }
    
    info_path = MODEL_OUTPUT_PATH / f'nn_training_info_fold{fold}.json'
    with open(info_path, 'w') as f:
        json.dump(training_info, f, indent=2)
    
    print(f"训练信息已保存到: {info_path}")
    
    return training_info

if 'model' in locals() and 'history' in locals():
    training_info = save_model_and_info(model, history, best_r2, fold=0)

## 13. 推理示例

In [ ]:
def predict_with_nn_model(test_df, model, batch_size=8192):
    """
    使用训练好的神经网络模型进行预测
    """
    model.eval()
    
    # 处理缺失值
    for col in CONFIG.FEATURE_COLS:
        if col in test_df.columns:
            test_df[col] = test_df[col].fillna(0)
    
    # 创建数据集
    class PredictDataset(Dataset):
        def __init__(self, df, feature_cols):
            self.df = df.reset_index(drop=True)
            self.feature_cols = feature_cols
        
        def __len__(self):
            return len(self.df)
        
        def __getitem__(self, idx):
            features = self.df.loc[idx, self.feature_cols].values.astype(np.float32)
            return torch.from_numpy(features)
    
    dataset = PredictDataset(test_df, CONFIG.FEATURE_COLS)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0
    )
    
    all_preds = []
    
    with torch.no_grad():
        for features in dataloader:
            features = features.to(DEVICE)
            predictions = model(features).cpu().numpy()
            all_preds.append(predictions)
    
    return np.concatenate(all_preds).ravel()

print("\n推理函数已定义")
print("\n使用示例:")
print("""
predictions = predict_with_nn_model(test_df, model)
result = pd.DataFrame({
    'row_id': test_df['row_id'],
    'responder_6': predictions
})
""")

## 14. 总结

In [ ]:
print("""
=== 神经网络训练总结 ===

1. 神经网络的优势:
   - 能够学习复杂的非线性关系
   - 自动特征学习
   - 可以在新数据上继续训练

2. 网络架构设计:
   - 输入层: 特征数量
   - 隐藏层: 逐步减小维度 (512 -> 256 -> 128)
   - 批归一化: 稳定训练
   - Dropout: 防止过拟合
   - 输出层: 单个值

3. 训练技巧:
   - 使用加权损失函数
   - 学习率调度
   - 早停机制
   - 梯度裁剪（可选）

4. 超参数调优建议:
   - 学习率: 1e-4 到 1e-3
   - 批大小: 4096 到 16384
   - 隐藏层维度: [256, 128, 64] 或 [512, 256, 128]
   - Dropout: 0.1 到 0.3
   - 权重衰减: 1e-5 到 1e-3

5. 下一步:
   - 尝试不同的网络架构
   - 进行超参数搜索
   - 与GBDT模型集成

6. Mac特别说明:
   - 如果有M1/M2芯片，可以考虑使用MPS加速
   - num_workers建议设为0避免多进程问题
   - pin_memory在Mac上通常不需要
""")